# Differentially Private Logistic Regression - Gallstone Dataset\n\nTrains a DP Logistic Regression classifier (via IBM diffprivlib) on the Gallstone dataset.


## 1. Import Libraries


In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from diffprivlib.models import LogisticRegression as DPLogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


## 2. Load Preprocessed Data


In [2]:
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

# DP norm precondition: clip every row to unit L2 norm (||x_i||_2 <= 1).
X_train = X_train / np.maximum(np.linalg.norm(X_train, axis=1, keepdims=True), 1.0)
X_test = X_test / np.maximum(np.linalg.norm(X_test, axis=1, keepdims=True), 1.0)


## 3. Load Baseline Results


In [3]:
try:
    baseline_df = pd.read_csv('output/baseline_results.csv')
    baseline_accuracy = baseline_df['accuracy'].values[0]
    baseline_f1 = baseline_df['f1_score'].values[0]
    print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
except FileNotFoundError:
    print("Warning: baseline_results.csv not found.")
    baseline_accuracy = None
    baseline_f1 = None


Baseline Accuracy: 0.7656


## 4. Full Sweep Across Epsilon Values


In [4]:
epsilon_values = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 4.0, 6.0, 8.0, 10.0]
N_RUNS = 30

results = {
    'epsilon': [], 'accuracy_mean': [], 'accuracy_std': [],
    'f1_score_mean': [], 'f1_score_std': [],
    'precision_mean': [], 'precision_std': [],
    'recall_mean': [], 'recall_std': [],
}

all_results = {
    'model_type': 'Differentially Private Logistic Regression',
    'dataset': 'Gallstone',
    'n_runs_per_epsilon': N_RUNS,
    'epsilon_results': {}
}

print(f"Training DP LR with {N_RUNS} runs per epsilon...")
print('=' * 80)

for eps in epsilon_values:
    run_accs, run_f1s, run_precs, run_recs, run_cms = [], [], [], [], []

    extra = ExtraMetrics()
    for run in range(N_RUNS):
        seed = run * 10 + 42
        dp_model = DPLogisticRegression(epsilon=eps, max_iter=1000, random_state=seed, data_norm=1.0)
        _t0 = time.perf_counter()
        dp_model.fit(X_train, y_train)
        train_time = time.perf_counter() - _t0
        y_pred = dp_model.predict(X_test)
        
        run_accs.append(accuracy_score(y_test, y_pred))
        extra.add(y_test, y_pred, model=dp_model, X=X_test, train_time=train_time)
        run_f1s.append(f1_score(y_test, y_pred, zero_division=0))
        run_precs.append(precision_score(y_test, y_pred, zero_division=0))
        run_recs.append(recall_score(y_test, y_pred, zero_division=0))
        run_cms.append(confusion_matrix(y_test, y_pred).tolist())
        if run == 0:
            import sys
            import os
            sys.path.append(os.path.abspath(os.path.join('..', '..')))
            from model_exporter import export_model
            export_model(dp_model, f'output/model/dp_lr_model_eps_{eps}.pkl')

    acc_mean, acc_std = np.mean(run_accs), np.std(run_accs)
    f1_mean, f1_std = np.mean(run_f1s), np.std(run_f1s)
    prec_mean, prec_std = np.mean(run_precs), np.std(run_precs)
    rec_mean, rec_std = np.mean(run_recs), np.std(run_recs)

    results['epsilon'].append(eps)
    # Merge in the added metrics; keys the notebook already tracks are skipped.
    for _key, _value in extra.summary().items():
        results.setdefault(_key, []).append(_value)
    results['accuracy_mean'].append(acc_mean)
    results['accuracy_std'].append(acc_std)
    results['f1_score_mean'].append(f1_mean)
    results['f1_score_std'].append(f1_std)
    results['precision_mean'].append(prec_mean)
    results['precision_std'].append(prec_std)
    results['recall_mean'].append(rec_mean)
    results['recall_std'].append(rec_std)

    all_results['epsilon_results'][str(eps)] = {
        'epsilon': float(eps), 'n_runs': N_RUNS,
        'accuracy_mean': acc_mean, 'accuracy_std': acc_std,
        'f1_mean': f1_mean, 'f1_std': f1_std,
        'precision_mean': prec_mean, 'precision_std': prec_std,
        'recall_mean': rec_mean, 'recall_std': rec_std,
        'per_run_accuracies': run_accs, 'per_run_f1s': run_f1s,
        'per_run_confusion_matrices': run_cms
    }
    all_results['epsilon_results'][str(eps)].update(extra.summary())
    all_results['epsilon_results'][str(eps)].update(extra.per_run())

    print(f"eps={eps:5.2f} | Acc: {acc_mean:.4f}+/-{acc_std:.4f} | F1: {f1_mean:.4f}+/-{f1_std:.4f}")

print('=' * 80)
print('Training complete!')


Training DP LR with 30 runs per epsilon...


Model successfully exported to output/model/dp_lr_model_eps_0.1.pkl
eps= 0.10 | Acc: 0.5047+/-0.0597 | F1: 0.4098+/-0.2373
Model successfully exported to output/model/dp_lr_model_eps_0.2.pkl
eps= 0.20 | Acc: 0.5042+/-0.0568 | F1: 0.4335+/-0.2049
Model successfully exported to output/model/dp_lr_model_eps_0.4.pkl
eps= 0.40 | Acc: 0.5120+/-0.0673 | F1: 0.4781+/-0.1439
Model successfully exported to output/model/dp_lr_model_eps_0.6.pkl


eps= 0.60 | Acc: 0.5151+/-0.0689 | F1: 0.4952+/-0.1162
Model successfully exported to output/model/dp_lr_model_eps_0.8.pkl
eps= 0.80 | Acc: 0.5068+/-0.0601 | F1: 0.4479+/-0.2059
Model successfully exported to output/model/dp_lr_model_eps_1.0.pkl
eps= 1.00 | Acc: 0.5188+/-0.0720 | F1: 0.4973+/-0.1348
Model successfully exported to output/model/dp_lr_model_eps_1.2.pkl


eps= 1.20 | Acc: 0.5203+/-0.0843 | F1: 0.5072+/-0.1176
Model successfully exported to output/model/dp_lr_model_eps_1.4.pkl
eps= 1.40 | Acc: 0.5297+/-0.0792 | F1: 0.5188+/-0.1044
Model successfully exported to output/model/dp_lr_model_eps_1.6.pkl
eps= 1.60 | Acc: 0.5307+/-0.0807 | F1: 0.5196+/-0.1015
Model successfully exported to output/model/dp_lr_model_eps_1.8.pkl


eps= 1.80 | Acc: 0.5359+/-0.0788 | F1: 0.5274+/-0.0954
Model successfully exported to output/model/dp_lr_model_eps_2.0.pkl
eps= 2.00 | Acc: 0.5406+/-0.0754 | F1: 0.5321+/-0.0909
Model successfully exported to output/model/dp_lr_model_eps_4.0.pkl
eps= 4.00 | Acc: 0.5995+/-0.0667 | F1: 0.5888+/-0.0775
Model successfully exported to output/model/dp_lr_model_eps_6.0.pkl


eps= 6.00 | Acc: 0.6292+/-0.0732 | F1: 0.6214+/-0.0809
Model successfully exported to output/model/dp_lr_model_eps_8.0.pkl
eps= 8.00 | Acc: 0.6604+/-0.0608 | F1: 0.6532+/-0.0666
Model successfully exported to output/model/dp_lr_model_eps_10.0.pkl
eps=10.00 | Acc: 0.6813+/-0.0544 | F1: 0.6769+/-0.0584
Training complete!


## 5. Save Results


In [5]:
os.makedirs('output', exist_ok=True)
results_df = pd.DataFrame(results)
if baseline_accuracy is not None:
    results_df['accuracy_loss_mean'] = 1 - results_df['accuracy_mean'] / baseline_accuracy
    results_df['accuracy_loss_pct'] = results_df['accuracy_loss_mean'] * 100

results_df.to_csv('output/dp_results.csv', index=False)
with open('output/dp_lr_report.json', 'w') as f:
    json.dump(all_results, f, indent=4)

print("DP results saved.")


DP results saved.
